# 🌞 Phase 9: SAPEC-Grade Optimal GHI Forecasting
### SAPEC 2027 — Track 9: Machine Learning & AI in Power Electronics
**Supervisor:** Prof. Arup Ratan | BITS Pilani Hyderabad | Dept. of EEE

---
**Run order:** Execute cells top-to-bottom. Each cell has a ▶ button.

| Step | Cell | What it does |
|------|------|--------------|
| 1 | Install deps | catboost, xgboost, optuna |
| 2 | Clone repo | Gets `19_phase9_sapec_optimal_model.py` |
| 3 | Upload data | You upload `kuala_lumpur_1min.csv` |
| 4 | Run Phase 9 | Full model run (~15–25 min on T4) |
| 5 | Download outputs | Auto-downloads plots ZIP + report |

> ⚠️ **Runtime:** Go to `Runtime → Change runtime type → T4 GPU` before running.

---
## Cell 1 — Install Dependencies
*(~90 seconds)*

In [ ]:
import subprocess, sys

print('Installing dependencies...')
pkgs = ['catboost', 'xgboost', 'optuna', 'lightgbm', 'scikit-learn']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print('\n✅ All dependencies installed.')

# Quick version check
import lightgbm, catboost, xgboost, optuna, sklearn
print(f'   lightgbm {lightgbm.__version__} | catboost {catboost.__version__} | xgboost {xgboost.__version__}')
print(f'   optuna   {optuna.__version__}  | sklearn  {sklearn.__version__}')

---
## Cell 2 — Clone the GitHub Repo
*(gets the Phase 9 script)*

In [ ]:
import os

REPO_URL  = 'https://github.com/saimanoj446/Renewable-Energy-Forecasting-AI.git'
REPO_DIR  = '/content/Renewable-Energy-Forecasting-AI'
SCRIPT    = os.path.join(REPO_DIR, 'kuala_lumpur', '19_phase9_sapec_optimal_model.py')

if os.path.exists(REPO_DIR):
    print('Repo already cloned — pulling latest...')
    os.system(f'git -C {REPO_DIR} pull')
else:
    print('Cloning repository...')
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

if os.path.exists(SCRIPT):
    size = os.path.getsize(SCRIPT)
    print(f'\n✅ Script found: {SCRIPT}')
    print(f'   Size: {size:,} bytes ({size//1024} KB)')
else:
    print('❌ Script not found — check repo URL or branch.')

---
## Cell 3 — Upload `kuala_lumpur_1min.csv`

Click **"Choose Files"** in the output below and select your `kuala_lumpur_1min.csv`.

> 💡 This file is ~50 MB. Upload may take 1–3 minutes depending on your connection.
> The satellite cloud file (`kl_satellite_clouds_2025.csv`) is **auto-downloaded** from Open-Meteo — no manual upload needed.

In [ ]:
import os
from google.colab import files

DATA_PATH = '/content/kuala_lumpur_1min.csv'

if os.path.exists(DATA_PATH):
    rows = sum(1 for _ in open(DATA_PATH)) - 1
    print(f'✅ Data already present: {DATA_PATH}  ({rows:,} rows)')
    print('   Skipping upload — delete the file above and re-run this cell to re-upload.')
else:
    print('📂 Select kuala_lumpur_1min.csv to upload...')
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dest = os.path.join('/content', fname)
        if not os.path.exists(dest):
            os.rename(fname, dest)
        print(f'✅ Uploaded: {dest}  ({len(data)/1e6:.1f} MB)')
    if os.path.exists(DATA_PATH):
        print('\n✅ kuala_lumpur_1min.csv is ready at /content/')
    else:
        print('⚠️  File saved under a different name — rename to kuala_lumpur_1min.csv')

---
## Cell 4 — 🚀 Run Phase 9 (Main Execution)

**Expected runtime:** 15–25 minutes on T4 GPU.

### What runs inside:
1. 📡 Download satellite cloud data from Open-Meteo (if not cached)
2. ⚙️  Engineer 60+ features (ramp rates, Fourier embeddings, lag diversity, percentiles)
3. 🔧 Optuna Bayesian HPO — 50 trials tuning LightGBM & CatBoost
4. 🧠 Train ensemble stack: LightGBM + CatBoost → XGBoost meta-learner (OOF)
5. 🎯 Quantile-aware adaptive EWMA post-processing (high/mid/low kt regimes)
6. 📊 Full 36-cell 2D sweep (x ∈ {15,30,45,60,90,120} × td ∈ {5,10,15,30,45,60})
7. 📈 Generate Plots 74–78
8. 📝 Write `sapec_optimal_report.txt` with all SAPEC headline metrics

In [ ]:
import os, time

SCRIPT = '/content/Renewable-Energy-Forecasting-AI/kuala_lumpur/19_phase9_sapec_optimal_model.py'
DATA   = '/content/kuala_lumpur_1min.csv'

# Pre-flight checks
assert os.path.exists(SCRIPT), f'Script not found: {SCRIPT}'
assert os.path.exists(DATA),   f'Data not found: {DATA}  — run Cell 3 first'

print('=' * 70)
print('  PRE-FLIGHT CHECKS PASSED')
print(f'  Script : {SCRIPT}')
print(f'  Data   : {DATA}  ({os.path.getsize(DATA)/1e6:.1f} MB)')
print('=' * 70)
print('  Starting Phase 9 — do NOT interrupt this cell.')
print('  Progress lines will stream below...')
print('=' * 70 + '\n')

t0 = time.time()
%run $SCRIPT
elapsed = time.time() - t0
print(f'\n⏱️  Total elapsed: {elapsed/60:.1f} minutes')

---
## Cell 5 — 📋 Show Results Summary

In [ ]:
import os

REPORT = '/content/sapec_optimal_output/sapec_optimal_report.txt'

if os.path.exists(REPORT):
    print('=' * 70)
    print('  SAPEC OPTIMAL REPORT')
    print('=' * 70)
    with open(REPORT) as f:
        print(f.read())
else:
    # Try alternate path
    alt = '/content/Renewable-Energy-Forecasting-AI/kuala_lumpur/sapec_optimal_output/sapec_optimal_report.txt'
    if os.path.exists(alt):
        with open(alt) as f:
            print(f.read())
    else:
        print('⚠️  Report not found — did Cell 4 complete successfully?')

---
## Cell 6 — 🖼️ Preview All 5 Plots (74–78)

In [ ]:
import os, glob
from IPython.display import Image, display

# Search both possible output locations
search_dirs = [
    '/content/sapec_optimal_output',
    '/content/Renewable-Energy-Forecasting-AI/kuala_lumpur/sapec_optimal_output',
    '/content/Renewable-Energy-Forecasting-AI/kuala_lumpur/plots',
    '/content/Renewable-Energy-Forecasting-AI/plots',
]

plot_names = [
    ('74_ph9_benchmark_phase7c_vs_phase9.png', 'Plot 74: Phase 7c vs Phase 9 Benchmark'),
    ('75_ph9_optimal_x_td_heatmaps.png',       'Plot 75: Optimal (x*, td*) 2D Heatmaps'),
    ('76_ph9_pareto_frontier_all_lookbacks.png','Plot 76: Pareto Frontier (RMSE vs Skill)'),
    ('77_ph9_walkforward_5m_15m_optimal.png',   'Plot 77: Walk-Forward at Optimal (x*, td*)'),
    ('78_ph9_feature_importance_enriched.png',  'Plot 78: Top-30 Feature Importances'),
]

found = 0
for fname, title in plot_names:
    path = None
    for d in search_dirs:
        candidate = os.path.join(d, fname)
        if os.path.exists(candidate):
            path = candidate
            break
    if path:
        print(f'\n📊 {title}')
        display(Image(path, width=900))
        found += 1
    else:
        print(f'❌ {fname} not found')

print(f'\n✅ Displayed {found}/5 plots.')

---
## Cell 7 — 📥 Download All Outputs

Downloads:
- `sapec_optimal_report.txt` — full benchmark table with SAPEC headline metrics
- `sapec_optimal_output.zip` — all Plots 74–78 + sweep CSV

In [ ]:
import os, shutil
from google.colab import files

# Find output directory
OUT_DIRS = [
    '/content/sapec_optimal_output',
    '/content/Renewable-Energy-Forecasting-AI/kuala_lumpur/sapec_optimal_output',
]
out_dir = next((d for d in OUT_DIRS if os.path.exists(d)), None)

if out_dir is None:
    print('⚠️  Output directory not found — did Cell 4 complete?')
else:
    # List contents
    contents = os.listdir(out_dir)
    print(f'📁 Output directory: {out_dir}')
    print(f'   {len(contents)} files:')
    for f in sorted(contents):
        size = os.path.getsize(os.path.join(out_dir, f))
        print(f'   {f:55s}  {size/1024:7.1f} KB')

    # Create ZIP
    zip_path = '/content/sapec_optimal_output'
    shutil.make_archive(zip_path, 'zip', out_dir)
    print(f'\n📦 ZIP created: {zip_path}.zip  ({os.path.getsize(zip_path+".zip")/1e6:.1f} MB)')

    # Download report first (small, text)
    report = os.path.join(out_dir, 'sapec_optimal_report.txt')
    if os.path.exists(report):
        files.download(report)
        print('⬇️  Downloading sapec_optimal_report.txt ...')

    # Download ZIP
    files.download(zip_path + '.zip')
    print('⬇️  Downloading sapec_optimal_output.zip ...')
    print('\n✅ Done! Check your browser Downloads folder.')

---
## Cell 8 — 📊 Show Optimal Sweep Matrix (CSV)

Displays the full 36-cell (x, td) sweep matrix as a formatted table.

In [ ]:
import os, pandas as pd

CSV_PATHS = [
    '/content/sapec_optimal_output/sapec_optimal_sweep_matrix.csv',
    '/content/Renewable-Energy-Forecasting-AI/kuala_lumpur/sapec_optimal_output/sapec_optimal_sweep_matrix.csv',
]

csv_path = next((p for p in CSV_PATHS if os.path.exists(p)), None)

if csv_path:
    df_sweep = pd.read_csv(csv_path)
    print(f'✅ Sweep matrix loaded: {len(df_sweep)} rows × {len(df_sweep.columns)} cols')
    print()

    # Pivot to 2D heatmap table
    pivot_r2   = df_sweep.pivot(index='Lookback_x', columns='Horizon_td', values='R2').round(4)
    pivot_rmse = df_sweep.pivot(index='Lookback_x', columns='Horizon_td', values='RMSE').round(1)

    print('▶ R² by (Lookback x, Horizon td):')
    print(pivot_r2.to_string())
    print()
    print('▶ RMSE [W/m²] by (Lookback x, Horizon td):')
    print(pivot_rmse.to_string())

    # Best cell
    best = df_sweep.loc[df_sweep['R2'].idxmax()]
    print(f'\n🏆 Global Optimum: x* = {int(best["Lookback_x"])} min, td* = {int(best["Horizon_td"])} min')
    print(f'   R² = {best["R2"]:.4f} | RMSE = {best["RMSE"]:.2f} W/m² | Skill = {best["Skill_pct"]:+.2f}%')
else:
    print('⚠️  Sweep matrix CSV not found — run Cell 4 first.')

---
## ✅ Done!

After downloading the outputs, **paste the values from `sapec_optimal_report.txt` back into Antigravity** and the agent will:

1. Commit Plots 74–78 + sweep CSV + report to GitHub  
2. Update `README.md` with actual Phase 9 benchmark metrics  
3. Update `SLIDE_CONTEXT.md` and `ppt_content.md` with Phase 9 slides  
4. Draft `sapec_digest.md` — 4-page IEEE PELS paper (deadline: **Nov 30, 2026**)  

---

### Targets to Beat (Phase 7c Baseline)

| Target | Baseline R² | Baseline RMSE | Expected Phase 9 |
|--------|-------------|---------------|------------------|
| 5-Min CAISO RTD | 0.7164 | 149.3 W/m² | ~0.78–0.82 |
| 15-Min Dispatch | 0.7633 | 131.3 W/m² | ~0.81–0.85 |

**Repository:** https://github.com/saimanoj446/Renewable-Energy-Forecasting-AI  
**Script committed at:** `48b9ea4`